# StealTheDealAI: Deep Neural Network Training

This notebook trains a PyTorch Residual Network on the processed PromptCloud dataset to predict prices.

**Instructions:**
1. Turn on the GPU in your Kaggle notebook (P100 or T4x2).
2. Upload the `training_data.csv` to Kaggle.
3. Run the cells below to train.
4. Download **both** `deep_neural_network.pth` and `dnn_norm_stats.json` from the Kaggle output directory and place them together in `StealDealProject/models/` - no manual code editing needed, `agents/deep_neural_network.py` loads the normalization stats from that file automatically.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
TRAINING_DATA_PATH = '/kaggle/input/datasets/damnyadav/amazon-30k/training_data.csv'

df = pd.read_csv(TRAINING_DATA_PATH)
df = df.dropna(subset=['description', 'price'])

print(f"Dataset size: {len(df)}")

Dataset size: 29339


In [3]:
# Prepare features
vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
X = vectorizer.transform(df['description'].astype(str)).toarray()

# Log-transform and standardize target (Price)
y_log = np.log1p(df['price'].values)
Y_MEAN = np.mean(y_log)
Y_STD = np.std(y_log)
y = (y_log - Y_MEAN) / Y_STD

print(f"Y_MEAN: {Y_MEAN}")
print(f"Y_STD: {Y_STD}")
print("COPY THESE VALUES TO deep_neural_network.py!")

Y_MEAN: 7.151269579629951
Y_STD: 1.2296543694352657
COPY THESE VALUES TO deep_neural_network.py!


In [4]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train).unsqueeze(1))
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test).unsqueeze(1))

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size, dropout_prob):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.block(x)
        out += residual
        return self.relu(out)

class DeepNeuralNetwork(nn.Module):
    def __init__(self, input_size, num_layers=10, hidden_size=4096, dropout_prob=0.2):
        super(DeepNeuralNetwork, self).__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
        )
        self.residual_blocks = nn.ModuleList()
        for i in range(num_layers - 2):
            self.residual_blocks.append(ResidualBlock(hidden_size, dropout_prob))
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.residual_blocks:
            x = block(x)
        return self.output_layer(x)

model = DeepNeuralNetwork(input_size=5000, hidden_size=2048).to(device) # Reduced hidden size for Kaggle T4

In [6]:
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_X.size(0)
    
    scheduler.step()
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_dataset):.4f} | Val Loss: {val_loss/len(test_dataset):.4f}")

Epoch 1/20 | Train Loss: 69.2384 | Val Loss: 0.6119
Epoch 2/20 | Train Loss: 0.4797 | Val Loss: 0.5255
Epoch 3/20 | Train Loss: 0.3805 | Val Loss: 0.4651
Epoch 4/20 | Train Loss: 0.3071 | Val Loss: 0.4532
Epoch 5/20 | Train Loss: 0.2202 | Val Loss: 0.4602
Epoch 6/20 | Train Loss: 0.1522 | Val Loss: 0.4378
Epoch 7/20 | Train Loss: 0.1028 | Val Loss: 0.4262
Epoch 8/20 | Train Loss: 0.0733 | Val Loss: 0.4190
Epoch 9/20 | Train Loss: 0.0573 | Val Loss: 0.4230
Epoch 10/20 | Train Loss: 0.0474 | Val Loss: 0.4254
Epoch 11/20 | Train Loss: 0.0385 | Val Loss: 0.4198
Epoch 12/20 | Train Loss: 0.0332 | Val Loss: 0.4135
Epoch 13/20 | Train Loss: 0.0284 | Val Loss: 0.4172
Epoch 14/20 | Train Loss: 0.0260 | Val Loss: 0.4154
Epoch 15/20 | Train Loss: 0.0225 | Val Loss: 0.4157
Epoch 16/20 | Train Loss: 0.0203 | Val Loss: 0.4147
Epoch 17/20 | Train Loss: 0.0193 | Val Loss: 0.4140
Epoch 18/20 | Train Loss: 0.0179 | Val Loss: 0.4140
Epoch 19/20 | Train Loss: 0.0176 | Val Loss: 0.4139
Epoch 20/20 | Train 

In [7]:
torch.save(model.state_dict(), 'deep_neural_network.pth')
print("Saved model to deep_neural_network.pth")
print("Download this file from the Kaggle output directory.")

Saved model to deep_neural_network.pth
Download this file from the Kaggle output directory.


In [8]:
import json

with open('dnn_norm_stats.json', 'w') as f:
    json.dump({"y_mean": float(Y_MEAN), "y_std": float(Y_STD)}, f)
print("Saved dnn_norm_stats.json")
print("Download BOTH deep_neural_network.pth and dnn_norm_stats.json from the Kaggle output")
print("directory and place them together in StealDealProject/models/ - agents/deep_neural_network.py")
print("loads Y_MEAN/Y_STD from this file automatically, no code edit needed.")

Saved dnn_norm_stats.json
Download BOTH deep_neural_network.pth and dnn_norm_stats.json from the Kaggle output
directory and place them together in StealDealProject/models/ - agents/deep_neural_network.py
loads Y_MEAN/Y_STD from this file automatically, no code edit needed.
